# Clasificación de audio con LSTM en Keras

## Objetivo

En este notebook veremos una aplicación adicional de modelos secuenciales: clasificación de audio.

La idea general será:

- cargar audios cortos,
- transformarlos a espectrogramas,
- interpretar cada espectrograma como una secuencia,
- y entrenar una red `LSTM` para clasificar comandos de voz simples.

> En este contexto, cada paso temporal de la secuencia contendrá información de frecuencias del audio.

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 1. Descargar un dataset de comandos de voz

Usaremos el dataset **mini speech commands** disponible en TensorFlow.

Contiene audios cortos asociados a palabras como:

- `yes`
- `no`
- `go`
- `stop`
- `up`
- `down`
- `left`
- `right`

In [ ]:
data_dir = tf.keras.utils.get_file(
    "mini_speech_commands.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/mini_speech_commands.zip",
    extract=True,
    cache_dir=".",
    cache_subdir="data"
)

data_dir = pathlib.Path(str(data_dir)).with_suffix("")
print("Ruta inicial:", data_dir)
print("Contenido inicial:", tf.io.gfile.listdir(str(data_dir)))

In [ ]:
data_dir = data_dir / "mini_speech_commands"

commands = np.array(tf.io.gfile.listdir(str(data_dir)))
commands = commands[(commands != "README.md") & (commands != ".DS_Store")]

print("Ruta final:", data_dir)
print("Clases:")
print(commands)

## 2. Crear conjuntos de entrenamiento y validación

Usaremos `audio_dataset_from_directory`, que simplifica mucho la carga de archivos `.wav`.

In [ ]:
seed = 42
batch_size = 32

train_ds, val_ds = tf.keras.utils.audio_dataset_from_directory(
    directory=data_dir,
    batch_size=batch_size,
    validation_split=0.2,
    seed=seed,
    output_sequence_length=16000,
    subset="both"
)

In [ ]:
label_names = np.array(train_ds.class_names)
print("Etiquetas:", label_names)

In [ ]:
for audio_batch, label_batch in train_ds.take(1):
    print("Shape batch de audio :", audio_batch.shape)
    print("Shape batch de labels:", label_batch.shape)

### Observación

Cada audio fue ajustado a longitud fija `16000`.

Eso corresponde, aproximadamente, a 1 segundo de audio muestreado a 16 kHz.

## 3. Visualizar un audio de ejemplo

In [ ]:
audio = audio_batch[0]
label = label_batch[0]

plt.figure(figsize=(10, 3))
plt.plot(audio.numpy())
plt.title(f"Forma de onda - clase: {label_names[label]}")
plt.xlabel("Muestra")
plt.ylabel("Amplitud")
plt.show()

### Escuchar el audio

Además de ver la forma de onda, también podemos escuchar el audio correspondiente.

In [ ]:
from IPython.display import Audio

Audio(audio.numpy().squeeze(), rate=16000)

## 4. De audio a espectrograma

No trabajaremos directamente con la onda cruda.

En cambio, convertiremos cada audio a un **espectrograma**, que resume cómo cambia el contenido en frecuencia a lo largo del tiempo.

Eso nos permite interpretar cada ejemplo como una **secuencia de vectores**.

In [34]:
def get_spectrogram(waveform):
    spectrogram = tf.signal.stft(
        waveform,
        frame_length=256,
        frame_step=128
    )
    spectrogram = tf.abs(spectrogram)
    return spectrogram

In [ ]:
print("Shape del audio original:", audio.shape)

In [ ]:
audio = tf.squeeze(audio, axis=-1)
spectrogram = get_spectrogram(audio)

print("Shape del espectrograma:", spectrogram.shape)

In [ ]:
plt.figure(figsize=(10, 4))
plt.imshow(
    tf.transpose(spectrogram).numpy(),
    aspect="auto",
    origin="lower"
)
plt.title(f"Espectrograma - clase: {label_names[label]}")
plt.xlabel("Paso temporal")
plt.ylabel("Frecuencia")
plt.colorbar()
plt.show()

### Idea clave

El espectrograma puede verse como una secuencia donde:

- el eje horizontal representa el tiempo,
- y en cada instante tenemos un vector de intensidades en distintas frecuencias.

Eso encaja muy bien con una `LSTM`.

## 5. Preparar los datasets de espectrogramas

In [38]:
def waveform_to_spectrogram(audio, label):
    audio = tf.squeeze(audio, axis=-1)
    spectrogram = get_spectrogram(audio)
    return spectrogram, label

In [39]:
train_spectrogram_ds = train_ds.map(waveform_to_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)
val_spectrogram_ds   = val_ds.map(waveform_to_spectrogram, num_parallel_calls=tf.data.AUTOTUNE)

In [40]:
train_spectrogram_ds = train_spectrogram_ds.prefetch(tf.data.AUTOTUNE)
val_spectrogram_ds   = val_spectrogram_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
for spec_batch, label_batch in train_spectrogram_ds.take(1):
    print("Shape batch espectrogramas:", spec_batch.shape)
    print("Shape labels:", label_batch.shape)

### Observación

El tensor ahora tiene forma:

- `batch_size`
- `timesteps`
- `frequency_bins`

Eso coincide con la estructura esperada por una LSTM:

`(batch, timesteps, features)`

## 6. Construcción del modelo LSTM

In [ ]:
for spec_batch, _ in train_spectrogram_ds.take(1):
    input_shape = spec_batch.shape[1:]

print("Input shape:", input_shape)

In [ ]:
num_classes = len(label_names)

model = keras.Sequential([
    layers.Input(shape=input_shape),
    layers.LSTM(64),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

model.summary()

### Interpretación del modelo

- `LSTM(64)` procesa la secuencia temporal del espectrograma.
- `Dense(64)` agrega una pequeña capa intermedia.
- La capa final `Dense(num_classes, softmax)` produce la clasificación.

In [44]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 7. Entrenamiento

In [ ]:
history = model.fit(
    train_spectrogram_ds,
    validation_data=val_spectrogram_ds,
    epochs=10,
    verbose=1
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy durante el entrenamiento")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss durante el entrenamiento")
plt.legend()
plt.show()

## 8. Evaluación rápida

In [ ]:
val_loss, val_acc = model.evaluate(val_spectrogram_ds, verbose=0)

print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_acc, 4))

## 9. Algunas predicciones

In [ ]:
for spec_batch, label_batch in val_spectrogram_ds.take(1):
    y_prob = model.predict(spec_batch, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    for i in range(5):
        print(f"Ejemplo {i+1}:")
        print("  Predicción :", label_names[y_pred[i]])
        print("  Etiqueta real:", label_names[label_batch[i]])
        print()

## 10. Conclusión

En este notebook vimos una aplicación de `LSTM` a audio.

La idea central fue:

- representar el audio como espectrograma,
- interpretar el espectrograma como secuencia,
- y usar una `LSTM` para clasificar.

Esto muestra que los modelos secuenciales no se limitan al texto: también pueden aplicarse a señales de audio.

## Bonus opcional: comparar con GRU

Como extensión, podemos reemplazar la capa `LSTM` por una capa `GRU`.

In [ ]:
gru_model = keras.Sequential([
    layers.Input(shape=input_shape),
    layers.GRU(64),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

gru_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

gru_model.summary()

In [ ]:
history_gru = gru_model.fit(
    train_spectrogram_ds,
    validation_data=val_spectrogram_ds,
    epochs=10,
    verbose=1
)

In [ ]:
gru_val_loss, gru_val_acc = gru_model.evaluate(val_spectrogram_ds, verbose=0)

print("Validation loss (GRU):", round(gru_val_loss, 4))
print("Validation accuracy (GRU):", round(gru_val_acc, 4))

## Bonus opcional 2: Bidirectional LSTM

También podemos procesar la secuencia en ambos sentidos usando `Bidirectional`.

In [ ]:
bilstm_model = keras.Sequential([
    layers.Input(shape=input_shape),
    layers.Bidirectional(layers.LSTM(64)),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

bilstm_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()

In [ ]:
history_bilstm = bilstm_model.fit(
    train_spectrogram_ds,
    validation_data=val_spectrogram_ds,
    epochs=10,
    verbose=1
)

In [ ]:
bilstm_val_loss, bilstm_val_acc = bilstm_model.evaluate(val_spectrogram_ds, verbose=0)

print("Validation loss (BiLSTM):", round(bilstm_val_loss, 4))
print("Validation accuracy (BiLSTM):", round(bilstm_val_acc, 4))